In [2]:
manifest_url = "https://eap.bl.uk/archive-file/EAP931-1-8-2/manifest?manifest=https%3A//eap.bl.uk/archive-file/EAP931-1-8-2/manifest"

In [3]:
# @title Fetch the manifest from EAP

import srsly
import requests

# The British Library blocks robots and other web crawlers from accessing their servers.  To signal that we are a human, we
# add a User-Agent to the request to show that we're a human. Do not share this with robots.
header = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/11"
}
manifest = requests.get(manifest_url, headers=header)
if manifest.status_code != 200:
    raise Exception(f"Error downloading manifest: {manifest.status_code}")

srsly.write_json("manifest.json", manifest.json())
# <-- In the files area to the left, you should now have a manifest.json file. Click on it to see the data.

manifest = manifest.json()

In [4]:
images = []
for canvas in manifest["sequences"][0]["canvases"]:
    for image in canvas["images"]:
        image_uri = image.get("resource", None).get("@id", None)
        if image_uri:
            images.append(image_uri)
print(f"Found {len(images)} images in manifest")

Found 448 images in manifest


In [5]:
import io
import base64
from PIL import Image
import aiohttp
import asyncio

async def load_image(image_url: str, session: aiohttp.ClientSession) -> str:
    """Async version of load_image with timeout and error handling"""
    try:
        async with session.get(image_url, timeout=aiohttp.ClientTimeout(total=30)) as response:
            print(f"Fetching image from {image_url}, status code: {response.status}")
            if response.status == 200:
                image_data = await response.read()

                # Convert image data to PIL Image
                pil_img = Image.open(io.BytesIO(image_data))

                # Convert pil Image to base64 data URI
                buf = io.BytesIO()
                pil_img.save(buf, format="PNG")
                encoded_string = base64.b64encode(buf.getvalue()).decode()
                return f"data:image/png;base64,{encoded_string}"
            else:
                print(f"Failed to fetch image: HTTP {response.status}")
                return None
    except asyncio.TimeoutError:
        print(f"Timeout loading image from {image_url}")
        return None
    except Exception as e:
        print(f"Error loading image from {image_url}: {e}")
        return None

In [1]:
# Install required async HTTP library
!pip install aiohttp


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import asyncio
import aiohttp
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

# Use AsyncOpenAI instead of OpenAI
client = AsyncOpenAI(
    api_key="psk-...",
    base_url="https://api.parasail.io/v1",
)

async def process_image(image_url: str, session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Process a single image with rate limiting and timeout"""
    async with semaphore:  # Limit concurrent requests
        print(f"Starting to process: {image_url}")
        
        # Load image with timeout
        image_uri = await load_image(image_url, session)
        if image_uri is None:
            print(f"Failed to load image from {image_url}")
            return None
        
        try:
            # Add timeout to API call as well
            completion = await asyncio.wait_for(
                client.chat.completions.create(
                    model="discircjipof-fantastic-futures",
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text",
                                    "text": "Extract text from the image. Preserve reading order. Return text as markdown."
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": image_uri
                                    }
                                }
                            ]
                        }
                    ],
                ),
                timeout=60  # 60 second timeout for API call
            )
            print(f"Successfully processed: {image_url}")
            return completion.choices[0].message
        except asyncio.TimeoutError:
            print(f"API timeout for image {image_url}")
            return None
        except Exception as e:
            print(f"Error processing image {image_url}: {e}")
            return None

async def process_all_images(images, max_concurrent=3):
    """Process all images concurrently with a limit on concurrent requests"""
    # Create aiohttp session for connection pooling
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    timeout = aiohttp.ClientTimeout(total=300)  # 5 minute total timeout
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        semaphore = asyncio.Semaphore(max_concurrent)
        
        # Create tasks for all images
        tasks = [process_image(image, session, semaphore) for image in images]
        
        # Process with progress bar
        results = []
        completed_count = 0
        
        for task in tqdm.as_completed(tasks, total=len(tasks), desc="Processing images"):
            try:
                result = await task
                completed_count += 1
                if result is not None:
                    results.append(result)
                print(f"Completed {completed_count}/{len(tasks)} images")
            except Exception as e:
                print(f"Task failed with error: {e}")
                completed_count += 1
    
    return results

# Run the async processing with reduced concurrency to avoid overwhelming the API
print(f"Starting to process {len(images)} images...")
output = await process_all_images(images, max_concurrent=3)

Starting to process 448 images...


Processing images:   0%|          | 0/448 [00:00<?, ?it/s]

Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/107.jp2/full/full/0/default.jpg
Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/213.jp2/full/full/0/default.jpg
Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/319.jp2/full/full/0/default.jpg
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/213.jp2/full/full/0/default.jpg, status code: 200
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/319.jp2/full/full/0/default.jpg, status code: 200
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/213.jp2/full/full/0/default.jpg, status code: 200
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/319.jp2/full/full/0/default.jpg, status code: 200
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/107.jp2/full/full/0/default.jpg, status code: 200
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/107.jp2/full/full/0/default.jpg, status code: 200


Processing images:   0%|          | 1/448 [00:16<2:06:35, 16.99s/it]

Successfully processed: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/319.jp2/full/full/0/default.jpg
Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/425.jp2/full/full/0/default.jpg
Completed 1/448 images


...

Successfully processed: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/109.jp2/full/full/0/default.jpg
Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/215.jp2/full/full/0/default.jpg
Completed 7/448 images
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/215.jp2/full/full/0/default.jpg, status code: 200
Fetching image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/215.jp2/full/full/0/default.jpg, status code: 200


Processing images:   2%|▏         | 7/448 [01:02<1:05:29,  8.91s/it]

Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/321.jp2/full/full/0/default.jpg


CancelledError: 

Error loading image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/321.jp2/full/full/0/default.jpg: 'NoneType' object has no attribute 'connect'
Failed to load image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/321.jp2/full/full/0/default.jpg
Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/427.jp2/full/full/0/default.jpg
Error loading image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/427.jp2/full/full/0/default.jpg: Session is closed
Failed to load image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/427.jp2/full/full/0/default.jpg
Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/110.jp2/full/full/0/default.jpg
Error loading image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/110.jp2/full/full/0/default.jpg: Session is closed
Failed to load image from https://images.eap.bl.uk/EAP931/EAP931_1_8_2/110.jp2/full/full/0/default.jpg
Starting to process: https://images.eap.bl.uk/EAP931/EAP931_1_8_2/216.jp2/full/full/0/default.jpg
Error l